# 00 - Data Preparation (run ONCE)

Builds the **frozen** train / val / test split from the three Kaggle datasets and writes `data_manifest.json`
(row counts + MD5 fingerprint of every split). Notebooks 01-04 read these files and **stop if the fingerprint
does not match**, so every model is guaranteed to use the identical test set.

**Run on Kaggle:** Add the 3 datasets as input (`nlp-dataset-1`, `-2`, `-3`). GPU is **not** needed (set Accelerator = None).
Then **Save Version > Save & Run All (Commit)**.

Afterwards: open the **Output** tab, take the 4 files inside `Chittagonian_AI_Project/` (`train_data.csv`, `val_data.csv`, `test_data.csv`, `data_manifest.json`) and save them as a Kaggle **Dataset** named `chittagonian-splits`.
Then add that dataset to each model notebook (01-04). Their path: `/kaggle/input/datasets/falaqueabrar69/chittagonian-splits`.
Run this notebook again only if the source data changes (then every model must be re-run).

The split logic is the one already used in your earlier Kaggle runs (grouped by the Bangla sentence, `random_state=42`), so the
test set is the same as before.

In [ ]:
# -- CELL 2: Multi-Source Dataset Ingestion & Sanitization ---------------
# Sources (auto-detected by column schema, see SCHEMAS):
#   ChatgaiyyaAlap  : cols  বাংলা | চট্টগ্রাম            (DOI: 10.17632/wtms9xbkkw.1)  main corpus + dictionary.csv
#   ANCHOLIK-NER    : cols  Standard_Bangla | Chittagong  "Regional_NER (Raw Sentences).csv"
#   ChattogramSent  : cols  Bengali | Chattogram | English | Sentiment (DOI: 10.17632/k6hts2ktxw.1)
# NOTE: ChattogramSent is ~90% a copy of ChatgaiyyaAlap (3,633 of 4,033 pairs identical).
#       It only contributes ~400 genuinely new pairs; they are kept, duplicates are dropped.
import pandas as pd
import os, glob
INPUT_DIRS = {
    'DS1': '/kaggle/input/datasets/falaqueabrar69/nlp-dataset-1',
    'DS2': '/kaggle/input/datasets/falaqueabrar69/nlp-dataset-2',
    'DS3': '/kaggle/input/datasets/falaqueabrar69/nlp-dataset-3',
}
PROJECT_DIR = '/kaggle/working/Chittagonian_AI_Project'

# -- Schema-based source discovery (does NOT depend on which input folder holds which file) --
SCHEMAS = {   # kind -> (standard-Bangla column, Chittagonian column)
    'chatgaiyya': ('বাংলা',           'চট্টগ্রাম'),   # ChatgaiyyaAlap main corpus (Dataset_Chittagong_2.0.csv)
    'dictionary': ('বাংলা',           'চট্টগ্রাম'),   # ChatgaiyyaAlap word lexicon (dictionary.csv)
    'ancholik':   ('Standard_Bangla', 'Chittagong'),  # ANCHOLIK-NER "Regional_NER (Raw Sentences).csv"
    'sentiment':  ('Bengali',         'Chattogram'),  # ChattogramSent (also has English, Sentiment)
}

def discover_sources(input_dirs):
    dirs = [d for d in input_dirs.values() if os.path.isdir(d)] or ['/kaggle/input']
    found, seen = {k: [] for k in SCHEMAS}, []
    for d in dirs:
        for p in sorted(glob.glob(os.path.join(d, '**', '*.csv'), recursive=True)):
            try:    cols = [str(c).strip() for c in pd.read_csv(p, nrows=0).columns]
            except Exception as e: seen.append((p, f'<unreadable: {e}>')); continue
            seen.append((p, cols))
            fname = os.path.basename(p).lower()
            if   {'Standard_Bangla', 'Chittagong'} <= set(cols): kind = 'ancholik'
            elif {'Bengali', 'Chattogram'}          <= set(cols): kind = 'sentiment'
            elif {'বাংলা', 'চট্টগ্রাম'}              <= set(cols): kind = 'dictionary' if 'dict' in fname else 'chatgaiyya'
            else: continue
            found[kind].append(p)
    return found, seen

def load_pairs(path, kind):
    sb, ct = SCHEMAS[kind]
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df[[sb, ct]].rename(columns={sb: 'standard_bangla', ct: 'chittagonian'})

def sanitize(df):
    """Strict, order-sensitive cleaning (NaN -> str -> 'nan' -> strip -> empty). Does NOT dedup."""
    df = df.dropna(subset=['standard_bangla', 'chittagonian']).copy()
    for c in ['standard_bangla', 'chittagonian']:
        df[c] = df[c].astype(str)
        df = df[df[c] != 'nan']
        df[c] = df[c].str.strip()
        df = df[df[c].str.len() > 0]
    return df

found, seen = discover_sources(INPUT_DIRS)
for kind, paths in found.items():
    print(f"{kind:<11}: {paths if paths else '- not found -'}")

# Translation sources are mandatory; the dictionary is optional (handled in Cell 3).
# Order = priority when the same Chittagonian sentence appears twice (first one wins).
ORDER = ['chatgaiyya', 'ancholik', 'sentiment']
parts, tags = [], []
for kind in ORDER:
    for p in found[kind]:
        d = sanitize(load_pairs(p, kind)); parts.append(d); tags.append(kind)
        print(f"  loaded {kind:<11} {len(d):>6} rows  <- {os.path.basename(p)}")
if not parts:
    msg = '\n'.join(f'  {p}  ->  {c}' for p, c in seen) or '  (no CSV files found - is Internet/dataset attached?)'
    raise FileNotFoundError(f'No translation CSV recognised. CSVs seen:\n{msg}')
if 'chatgaiyya' not in tags:
    print('[WARN] ChatgaiyyaAlap main corpus not found - training on the remaining sources only.')

merged_df = pd.concat(parts, ignore_index=True)
before = len(merged_df)

# Dedup on the Chittagonian side: the same Chittagonian sentence must never appear twice
# (72 sentences exist with slightly different Bangla targets across sources -> would leak across splits).
merged_df = merged_df.drop_duplicates(subset=['chittagonian'], keep='first').reset_index(drop=True)

print(f"Raw combined: {before} | After sanitization + dedup: {len(merged_df)} rows")
merged_df.to_csv('final_merged_chittagonian.csv', index=False)
# (Kaggle: file is saved in /kaggle/working - no browser download)

In [ ]:
# -- CELL 3: Leakage-Free Train/Val/Test Split & Holdout-Filtered Dict ---
import pandas as pd, shutil, os, glob
from sklearn.model_selection import GroupShuffleSplit
INPUT_DIRS = {
    'DS1': '/kaggle/input/datasets/falaqueabrar69/nlp-dataset-1',
    'DS2': '/kaggle/input/datasets/falaqueabrar69/nlp-dataset-2',
    'DS3': '/kaggle/input/datasets/falaqueabrar69/nlp-dataset-3',
}
PROJECT_DIR = '/kaggle/working/Chittagonian_AI_Project'

# -- Schema-based source discovery (does NOT depend on which input folder holds which file) --
SCHEMAS = {   # kind -> (standard-Bangla column, Chittagonian column)
    'chatgaiyya': ('বাংলা',           'চট্টগ্রাম'),   # ChatgaiyyaAlap main corpus (Dataset_Chittagong_2.0.csv)
    'dictionary': ('বাংলা',           'চট্টগ্রাম'),   # ChatgaiyyaAlap word lexicon (dictionary.csv)
    'ancholik':   ('Standard_Bangla', 'Chittagong'),  # ANCHOLIK-NER "Regional_NER (Raw Sentences).csv"
    'sentiment':  ('Bengali',         'Chattogram'),  # ChattogramSent (also has English, Sentiment)
}

def discover_sources(input_dirs):
    dirs = [d for d in input_dirs.values() if os.path.isdir(d)] or ['/kaggle/input']
    found, seen = {k: [] for k in SCHEMAS}, []
    for d in dirs:
        for p in sorted(glob.glob(os.path.join(d, '**', '*.csv'), recursive=True)):
            try:    cols = [str(c).strip() for c in pd.read_csv(p, nrows=0).columns]
            except Exception as e: seen.append((p, f'<unreadable: {e}>')); continue
            seen.append((p, cols))
            fname = os.path.basename(p).lower()
            if   {'Standard_Bangla', 'Chittagong'} <= set(cols): kind = 'ancholik'
            elif {'Bengali', 'Chattogram'}          <= set(cols): kind = 'sentiment'
            elif {'বাংলা', 'চট্টগ্রাম'}              <= set(cols): kind = 'dictionary' if 'dict' in fname else 'chatgaiyya'
            else: continue
            found[kind].append(p)
    return found, seen

def load_pairs(path, kind):
    sb, ct = SCHEMAS[kind]
    df = pd.read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df[[sb, ct]].rename(columns={sb: 'standard_bangla', ct: 'chittagonian'})

def sanitize(df):
    """Strict, order-sensitive cleaning (NaN -> str -> 'nan' -> strip -> empty). Does NOT dedup."""
    df = df.dropna(subset=['standard_bangla', 'chittagonian']).copy()
    for c in ['standard_bangla', 'chittagonian']:
        df[c] = df[c].astype(str)
        df = df[df[c] != 'nan']
        df[c] = df[c].str.strip()
        df = df[df[c].str.len() > 0]
    return df

project_dir = PROJECT_DIR
os.makedirs(project_dir, exist_ok=True)

# -- Main data: output of Cell 2 --------------------------------------------
if not os.path.exists('final_merged_chittagonian.csv'):
    raise FileNotFoundError("final_merged_chittagonian.csv not found - run Cell 2 first.")
main_df = sanitize(pd.read_csv('final_merged_chittagonian.csv'))
main_df = main_df.drop_duplicates(subset=['chittagonian']).reset_index(drop=True)

# -- Dictionary (optional): ChatgaiyyaAlap dictionary.csv, picked by schema + filename --
found, _ = discover_sources(INPUT_DIRS)
if found['dictionary']:
    dict_path = found['dictionary'][0]
    dict_df = sanitize(load_pairs(dict_path, 'dictionary'))
    print('Dictionary:', dict_path, f'({len(dict_df)} rows)')
else:
    dict_df = pd.DataFrame(columns=['standard_bangla', 'chittagonian'])
    print('[WARN] dictionary.csv not found in any input dataset - continuing WITHOUT dictionary pairs.')

# -- 70 / 15 / 15 split, grouped by the Bangla side -------------------------
# Same Bangla sentence in train and test (e.g. different dialect rows / sources) would leak the target.
def gsplit(df, test_size):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=42)
                  .split(df, groups=df['standard_bangla']))
    return df.iloc[tr].reset_index(drop=True), df.iloc[te].reset_index(drop=True)

train_df, temp_df = gsplit(main_df, 0.30)
val_df,   test_df = gsplit(temp_df, 0.50)

# -- Holdout-filtered dictionary (filter BEFORE append) ---------------------
holdout_ctg = set(val_df['chittagonian']) | set(test_df['chittagonian'])
holdout_sb  = set(val_df['standard_bangla']) | set(test_df['standard_bangla'])
dict_df = dict_df[~dict_df['chittagonian'].isin(holdout_ctg) &
                  ~dict_df['standard_bangla'].isin(holdout_sb)]
dict_df = dict_df.drop_duplicates(subset=['chittagonian'])
dict_df = dict_df[~dict_df['chittagonian'].isin(set(train_df['chittagonian']))]

# -- Build final train set ---------------------------------------------------
final_train_df = pd.concat([train_df, dict_df], ignore_index=True)
final_train_df = final_train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# -- Strict programmatic leakage assertions ---------------------------------
assert len(set(final_train_df['chittagonian'])    & holdout_ctg) == 0, "Leakage: Chittagonian overlap train vs val/test!"
assert len(set(final_train_df['standard_bangla']) & holdout_sb)  == 0, "Leakage: Bangla-target overlap train vs val/test!"
assert len(set(val_df['chittagonian']) & set(test_df['chittagonian'])) == 0, "Leakage: val vs test overlap!"
print("[OK] Leakage assertions passed - zero overlap confirmed.")

final_train_df.to_csv('train_data.csv', index=False)
val_df.to_csv('val_data.csv',           index=False)
test_df.to_csv('test_data.csv',         index=False)

for f in ['train_data.csv', 'val_data.csv', 'test_data.csv']:
    shutil.copy(f, project_dir)

print(f"Train: {len(final_train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Dictionary pairs added to training: {len(dict_df)} (holdout-filtered)")

In [ ]:
# -- CELL 3: Manifest (fingerprints) + leakage checks + dataset statistics -------------
import os, json, hashlib, unicodedata
import pandas as pd
project_dir = '/kaggle/working/Chittagonian_AI_Project'
SEED = 42
split_paths = {s: f'{project_dir}/{s}_data.csv' for s in ['train', 'val', 'test']}

def md5(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

splits = {s: pd.read_csv(p, keep_default_na=False) for s, p in split_paths.items()}

def overlaps(col):
    a, b, c = (set(splits[s][col]) for s in ['train', 'val', 'test'])
    return {"train&val": len(a & b), "train&test": len(a & c), "val&test": len(b & c)}
leaks_ctg, leaks_bn = overlaps('chittagonian'), overlaps('standard_bangla')
print("Chittagonian-side overlap:", leaks_ctg)
print("Bangla-side overlap      :", leaks_bn)
assert leaks_ctg["train&val"] == 0 and leaks_ctg["train&test"] == 0, "Data leakage (Chittagonian) into training set!"
assert leaks_bn["train&val"] == 0 and leaks_bn["train&test"] == 0, "Data leakage (Bangla target) into training set!"
if leaks_ctg["val&test"] or leaks_bn["val&test"]:
    print("[WARN] Some sentences appear in both val and test (no training leak, but report it).")

nfc_changed = int(sum(unicodedata.normalize('NFC', t) != t
                      for s in splits.values() for col in ['chittagonian', 'standard_bangla'] for t in s[col]))
print(f"Cells that NFC normalisation would change: {nfc_changed}")

def length_stats(series):
    n = series.str.split().str.len()
    return {"mean_words": round(float(n.mean()), 2), "median_words": float(n.median()), "max_words": int(n.max())}

manifest = {s: {"rows": len(df), "md5": md5(split_paths[s]),
                "src_len": length_stats(df['chittagonian']),
                "tgt_len": length_stats(df['standard_bangla'])} for s, df in splits.items()}
_found = globals().get('found') or {}
manifest.update({"seed": SEED,
                 "split_ratio": "70/15/15, grouped by Bangla sentence (dictionary added to train only)",
                 "overlap_chittagonian": leaks_ctg, "overlap_bangla": leaks_bn, "nfc_changed_cells": nfc_changed,
                 "sources_found": {k: [os.path.basename(p) for p in v] for k, v in _found.items()},
                 "merged_rows_after_dedup": int(len(merged_df)) if 'merged_df' in globals() else None})

with open(f'{project_dir}/data_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
for s in ['train', 'val', 'test']:
    m = manifest[s]
    print(f"{s:<5}: {m['rows']:>6} rows | MD5 {m['md5'][:10]}... | "
          f"src {m['src_len']['mean_words']} words avg | tgt {m['tgt_len']['mean_words']} words avg")
print("=" * 60)
print("[OK] data_manifest.json written. Copy these row counts into the paper's dataset table.")
print("Next: Save Version > Save & Run All, then add this notebook's OUTPUT as an input to notebooks 01-04.")